# Baseline 2 — Exp07: Two-View Test-Time Augmentation (Development Only)

**Goal:** Test whether averaging predictions from an image and its horizontal flip improves the existing **Exp04** multi-task EfficientNet-B0 model. **No retraining**, architecture change, or official test data.

**Two pre-declared inference variants:** original validation image, and the same tensor flipped horizontally (`torch.flip(..., dims=[-1])`). Average their **Crop logits** and **Disease logits** with equal weight, then use the existing `composition_scores()` and all **9,625 candidate pairs**.

**Four transparent comparisons:** Exp04 original (γ=0), Exp07 TTA only (γ=0), Exp06 calibration only (γ=4.0), and Exp07 TTA + fixed γ=4.0. Gamma 4.0 is **reused from Exp06**; do not tune new gamma values in Exp07. **Exp07 TTA-only is the primary new experiment.**

This is a development-only, post-processing comparison. The test sets must remain untouched. Keep `RUN_EXP07 = False` until you explicitly decide to run the Controls cell.

This reference copy has inference disabled by default.

## 1. Imports and safe configuration

Place this notebook at `C:\AML\ML_Project\Baseline2`, beside the existing `model.py`, `evaluate.py` and `splits_v2` directory. All data loaders use only the named **development split files**.

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

# Locate the project before importing its modules, including from nested notebook folders.
kernel_dir = Path.cwd().resolve()
PROJECT_DIR = next(
    (folder for folder in (kernel_dir, *kernel_dir.parents)
     if all((folder / name).is_file() for name in ("baseline2.ipynb", "model.py", "evaluate.py"))),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Start Jupyter in the Baseline2 folder or one of its subfolders.")
sys.path.insert(0, str(PROJECT_DIR))

from model import load_checkpoint
from evaluate import (
    DPDDataset, validate_development_splits, validation_transform,
    composition_scores, make_candidates, harmonic_mean,
    require_candidate_coverage, sha256_file,
)

RUN_EXP07 = False

SPLIT_DIR = PROJECT_DIR / "splits_v2"
IMAGE_ROOT = Path(os.environ.get(
    "B2_IMAGE_ROOT", str(PROJECT_DIR.parent / "DPD_Small_dataset")
)).expanduser()
IMAGE_ROOT = (PROJECT_DIR / IMAGE_ROOT).resolve()
CHECKPOINT_PATH = PROJECT_DIR / "outputs" / "exp04" / "best_development.pt"
EXP04_METRICS_PATH = PROJECT_DIR / "outputs" / "exp04" / "validation_metrics.json"
EXP06_REPORT_PATH = PROJECT_DIR / "outputs" / "exp06_calibration" / "calibration_summary.json"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "exp07_tta"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
NUM_WORKERS = 2     # You can change to 0 if Windows/Jupyter workers fail; no training.
FIXED_GAMMA = 4.0   # PREVIOUS Exp06 development-selected value: do not re-tune on Exp07.

assert (PROJECT_DIR / "model.py").is_file(), "model.py not found; check PROJECT_DIR"
assert (PROJECT_DIR / "evaluate.py").is_file(), "evaluate.py not found; check PROJECT_DIR"
assert FIXED_GAMMA == 4.0, "This comparison must retain Exp06's predeclared gamma=4.0"
print("Device:", DEVICE)
print("Exp04 checkpoint:", CHECKPOINT_PATH)
print("Exp07 output:", OUTPUT_DIR)
print("No training; no official test access.")

## 2. Load Exp04 and verify development split and candidate policy

This is essential: **Exp07 must evaluate the exact same 695 seen and 400 unseen images** as Exp04/Exp06 and use the same 9,625 ordered candidates. Seen-pair mask is made from `train_fit.csv` **only**.

In [ ]:
if RUN_EXP07:
    frames, split_report = validate_development_splits(SPLIT_DIR)
    model, ck = load_checkpoint(CHECKPOINT_PATH, DEVICE)
    model.eval()
    assert ck.get("stage") == 1, "Expected Exp04 Stage 1 checkpoint"
    assert ck.get("config", {}).get("experiment_id") == "exp04", "Selected checkpoint is not Exp04"
    assert split_report["sha256"] == ck["development_split"]["sha256"], "Development split differs from Exp04"

    pair_candidates = torch.as_tensor(ck["candidate_pairs"], dtype=torch.long)
    assert len(pair_candidates) == 9625 and torch.equal(pair_candidates, make_candidates("all_pairs")), "Candidate policy/order mismatch"
    require_candidate_coverage(pair_candidates, [frames["train_fit"], frames["val_seen"], frames["val_unseen"]])

    train_pairs = set(map(tuple, frames["train_fit"][["crop_id", "disease_id"]].to_numpy(dtype=int).tolist()))
    seen_pair_mask = torch.tensor([tuple(p) in train_pairs for p in pair_candidates.tolist()], dtype=torch.bool)
    assert int(seen_pair_mask.sum()) == 313, "Expected 313 train_fit-seen compositions"
    assert all(tuple(p) not in train_pairs for p in frames["val_unseen"][["crop_id", "disease_id"]].to_numpy(dtype=int).tolist())
    assert all(tuple(p) in train_pairs for p in frames["val_seen"][["crop_id", "disease_id"]].to_numpy(dtype=int).tolist())
    print(f"Development: {len(frames['train_fit']):,} train / {len(frames['val_seen'])} seen / {len(frames['val_unseen'])} unseen")
    print(f"Candidate pairs: {len(pair_candidates):,}; train_fit-seen pairs: {int(seen_pair_mask.sum())}")
    print("Exp04, splits, candidate ordering and train-only mask verified.")

## 3. Synthetic TTA self-test (no real images)

A horizontal flip must keep the image size and values, and two-view aggregation must equal the arithmetic mean. This checks the method before any development inference.

In [ ]:
def average_two_view_logits(original_logits, flipped_logits):
    if original_logits.shape != flipped_logits.shape:
        raise ValueError("TTA view logits have different shapes")
    return (original_logits.float() + flipped_logits.float()) / 2.0

def apply_fixed_calibration(scores, gamma):
    if not np.isfinite(gamma) or gamma < 0:
        raise ValueError("Gamma must be finite and >= 0")
    if scores.ndim != 2 or scores.shape[1] != len(seen_pair_mask):
        raise ValueError("Candidate-score shape mismatch")
    return scores - float(gamma) * seen_pair_mask.to(scores.dtype)[None, :]

toy_image = torch.arange(2 * 3 * 4 * 5, dtype=torch.float32).reshape(2, 3, 4, 5)
assert torch.equal(torch.flip(torch.flip(toy_image, dims=[-1]), dims=[-1]), toy_image)
a = torch.tensor([[2.0, -1.0, 3.0]])
b = torch.tensor([[0.0, 3.0, 1.0]])
assert torch.allclose(average_two_view_logits(a, b), torch.tensor([[1.0, 1.0, 2.0]]))
if RUN_EXP07:
    scores = torch.zeros(2, len(pair_candidates), dtype=torch.float32)
    assert torch.equal(scores, apply_fixed_calibration(scores, 0.0))
    assert torch.all(apply_fixed_calibration(scores, 4.0)[:, seen_pair_mask] == -4.0)
    assert torch.all(apply_fixed_calibration(scores, 4.0)[:, ~seen_pair_mask] == 0.0)
print("Synthetic two-view check passed.")

## 4. Define inference and evaluation functions — definitions ONLY

Two sequential forward passes per batch avoid loading both views into GPU simultaneously. We first compute **Exp04 original logits** and **Exp07 averaged logits**, and then evaluate pair rankings under γ=0 or γ=4.0. All metrics are development-only.

In [ ]:
@torch.inference_mode()
def collect_two_view_scores(split_name):
    if split_name not in ("val_seen", "val_unseen"):
        raise ValueError("Development validation splits only")
    dataset = DPDDataset(frames[split_name], IMAGE_ROOT, validation_transform())
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    model.eval()
    cand_dev = pair_candidates.to(DEVICE)
    saved = {"raw_pair": [], "tta_pair": [], "raw_crop": [], "tta_crop": [],
             "raw_disease": [], "tta_disease": [], "targets": [], "filenames": []}

    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            orig_out = model(images)
            flip_out = model(torch.flip(images, dims=[-1]))
        raw_crop = orig_out["crop_logits"].float()
        raw_disease = orig_out["disease_logits"].float()
        tta_crop = average_two_view_logits(raw_crop, flip_out["crop_logits"])
        tta_disease = average_two_view_logits(raw_disease, flip_out["disease_logits"])

        saved["raw_pair"].append(composition_scores(raw_crop, raw_disease, cand_dev).cpu())
        saved["tta_pair"].append(composition_scores(tta_crop, tta_disease, cand_dev).cpu())
        for name, value in (("raw_crop", raw_crop), ("tta_crop", tta_crop),
                            ("raw_disease", raw_disease), ("tta_disease", tta_disease)):
            saved[name].append(value.cpu())
        saved["targets"].append(torch.stack([batch["crop"], batch["disease"]], dim=1).long())
        saved["filenames"].extend(batch["filename"])

    for key in saved:
        if key != "filenames":
            saved[key] = torch.cat(saved[key], dim=0)
    assert len(saved["filenames"]) == len(dataset)
    assert saved["raw_pair"].shape == saved["tta_pair"].shape == (len(dataset), 9625)
    assert saved["targets"].shape == (len(dataset), 2)
    return saved

def evaluate_pair_scores(pair_scores, targets, *, gamma=0.0):
    scores = apply_fixed_calibration(pair_scores, gamma)
    pair_ids = scores.topk(5, dim=1).indices
    ranked_pairs = pair_candidates[pair_ids]
    match = (ranked_pairs == targets[:, None, :]).all(dim=2)
    top1 = float(match[:, 0].float().mean())
    top5 = float(match.any(dim=1).float().mean())
    seen_rate = float(seen_pair_mask[pair_ids[:, 0]].float().mean())
    return {"top1": top1, "top5": top5, "predicted_train_seen_pair_rate": seen_rate}, ranked_pairs[:, 0]

def task_top1(data, variant, task):
    logits = data[f"{variant}_{task}"]
    idx = 0 if task == "crop" else 1
    return float((logits.argmax(1) == data["targets"][:, idx]).float().mean())

def evaluate_variants(s, u):
    variants = [
        ("exp04_original", "raw", 0.0),
        ("exp07_tta_only", "tta", 0.0),
        ("exp06_calibration_gamma4", "raw", FIXED_GAMMA),
        ("exp07_tta_plus_gamma4", "tta", FIXED_GAMMA),
    ]
    rows, predictions = [], {}
    for label, source, gamma in variants:
        sm, sp = evaluate_pair_scores(s[f"{source}_pair"], s["targets"], gamma=gamma)
        um, up = evaluate_pair_scores(u[f"{source}_pair"], u["targets"], gamma=gamma)
        rows.append({"method": label, "gamma": gamma,
                     "seen_top1": sm["top1"], "seen_top5": sm["top5"],
                     "unseen_top1": um["top1"], "unseen_top5": um["top5"],
                     "harmonic_mean": harmonic_mean(sm["top1"], um["top1"]),
                     "seen_predicted_train_pair_rate": sm["predicted_train_seen_pair_rate"],
                     "unseen_predicted_train_pair_rate": um["predicted_train_seen_pair_rate"],
                     "seen_crop_top1": task_top1(s, source, "crop"),
                     "unseen_crop_top1": task_top1(u, source, "crop"),
                     "seen_disease_top1": task_top1(s, source, "disease"),
                     "unseen_disease_top1": task_top1(u, source, "disease")})
        predictions[(label,"val_seen")] = sp
        predictions[(label,"val_unseen")] = up
    return pd.DataFrame(rows), predictions

print("Two-view TTA inference and evaluation functions defined. No dataset inference started.")

## 5. Development inference control

RUN_EXP07 defaults to False in the configuration cell. The historical workflow requires the local Exp04 checkpoint and original development splits, and refuses to overwrite the saved experiment directory.

In [ ]:
if RUN_EXP07:
    if OUTPUT_DIR.exists():
        raise FileExistsError(f"Refusing to overwrite previous Exp07 results: {OUTPUT_DIR}")
    start = time.perf_counter()
    print("Exp07: Collecting original + horizontally flipped predictions (no training)...")
    seen = collect_two_view_scores("val_seen")
    unseen = collect_two_view_scores("val_unseen")
    comparison, predictions = evaluate_variants(seen, unseen)

    # Must recover Exp04's previously verified predictions before writing any output.
    base = comparison.set_index("method").loc["exp04_original"]
    if EXP04_METRICS_PATH.exists():
        reference = json.loads(EXP04_METRICS_PATH.read_text(encoding="utf-8"))
        expected_seen = float(reference["seen"]["composition_top1"])
        expected_unseen = float(reference["unseen"]["composition_top1"])
    else:
        expected_seen, expected_unseen = (0.6647482014388489, 0.0225)  # Verified Exp04 development values.
    assert abs(float(base["seen_top1"]) - expected_seen) < 1e-5, "Exp04 seen baseline mismatch"
    assert abs(float(base["unseen_top1"]) - expected_unseen) < 1e-5, "Exp04 unseen baseline mismatch"

    # Exp06's gamma=4 result must also agree if its report is available.
    if EXP06_REPORT_PATH.exists():
        prior = json.loads(EXP06_REPORT_PATH.read_text(encoding="utf-8"))
        assert abs(float(prior["selected_on_development"]["gamma"]) - FIXED_GAMMA) < 1e-9
        cal = comparison.set_index("method").loc["exp06_calibration_gamma4"]
        for metric in ("seen_top1", "unseen_top1", "harmonic_mean"):
            assert abs(float(cal[metric]) - float(prior["selected_on_development"][metric])) < 1e-5, (
                "Exp06 gamma=4 mismatch: " + metric)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    comparison.to_csv(OUTPUT_DIR / "method_comparison.csv", index=False)
    methods = comparison.set_index("method").to_dict(orient="index")
    summary = {
        "experiment": "exp07_tta", "parent": "exp04", "stage": "development_postprocessing_only",
        "tta": {"views": ["original", "horizontal_flip"], "logit_average": "equal_weight_0.5_each",
                "composition_scoring": "sum_of_crop_and_disease_log_softmax"},
        "candidate_mode": "all_pairs", "candidate_count": 9625,
        "train_seen_pair_count": int(seen_pair_mask.sum()), "comparison_methods": methods,
        "gamma4_origin": "Fixed Exp06 development-selected gamma, not re-tuned by Exp07",
        "primary_comparison": "exp04_original vs exp07_tta_only (both gamma=0)",
        "split_sha256": split_report["sha256"], "checkpoint_sha256": sha256_file(CHECKPOINT_PATH),
        "training_performed": False, "official_test_accessed": False,
        "selection_caveat": "Development estimates only; gamma=4 was chosen on the same development data in Exp06."
    }
    (OUTPUT_DIR / "tta_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    for label, data in (("val_seen", seen), ("val_unseen", unseen)):
        target_np = data["targets"].numpy()
        for method in comparison["method"]:
            pred_np = predictions[(method, label)].numpy()
            pd.DataFrame({
                "filename": data["filenames"], "true_crop_id": target_np[:, 0],
                "true_disease_id": target_np[:, 1], "pred_crop_id": pred_np[:, 0],
                "pred_disease_id": pred_np[:, 1], "correct": (pred_np == target_np).all(axis=1),
                "pred_is_train_seen_pair": [tuple(p) in train_pairs for p in pred_np.tolist()],
            }).to_csv(OUTPUT_DIR / f"{label}_{method}_predictions.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    xs = np.arange(len(comparison)); names = ["Exp04", "TTA", "Exp06 γ=4", "TTA+γ=4"]
    width = 0.36
    axes[0].bar(xs - width/2, comparison["seen_top1"] * 100, width, label="Seen")
    axes[0].bar(xs + width/2, comparison["unseen_top1"] * 100, width, label="Unseen")
    axes[0].set(ylabel="Composition Top-1 (%)", title="Development seen vs unseen")
    axes[0].set_xticks(xs, names, rotation=20)
    axes[0].legend()
    axes[1].bar(xs, comparison["harmonic_mean"] * 100)
    axes[1].set(ylabel="Harmonic Mean (%)", title="Development HM comparison")
    axes[1].set_xticks(xs, names, rotation=20)
    fig.tight_layout(); fig.savefig(OUTPUT_DIR / "tta_comparison.png", dpi=170)
    plt.show(); plt.close(fig)

    display(comparison[["method", "seen_top1", "unseen_top1", "harmonic_mean",
                        "seen_crop_top1", "unseen_crop_top1", "seen_disease_top1", "unseen_disease_top1"]]
            .style.format({k: "{:.2%}" for k in (
                "seen_top1", "unseen_top1", "harmonic_mean", "seen_crop_top1",
                "unseen_crop_top1", "seen_disease_top1", "unseen_disease_top1")}))
    print("Exp07 complete in", round(time.perf_counter()-start, 1), "seconds. Saved:", OUTPUT_DIR)
    print("Main comparison: Exp04 original vs TTA-only. No training or official test performed.")
else:
    print("Exp07 OFF: no inference/training. Set RUN_EXP07=True ONLY in this cell when ready.")

## 6. Recorded result

Two-view TTA matched Exp04’s 4.35% harmonic mean. TTA with the fixed calibration penalty yielded 10.22%, compared with 11.40% for calibration alone. The official workflow used a single view.

See the [saved Exp07 report](../../outputs/development/exp07/tta_summary.json).